# Venmito Data Cleaning

Addresses the six findings from `venmito_data_diagnostics.ipynb`.

**Nothing is dropped.** Every defect becomes a flag column, so downstream code filters by
intent (`is_clean`, `needs_review`) and the raw evidence stays auditable.

| # | Finding | Treatment |
|---|---|---|
| 1 | People split across two incompatible schemas | Normalize + outer-join on id |
| 2 | id 998 ambiguous, Fey Kuser duplicated | Explicit policy, no silent overwrite |
| 3 | Three different join keys | phone→id / email→id lookups |
| 4 | Duplicate promotion ids | Re-key on row position |
| 5 | Bad prices, duplicate transaction | Recompute + flag + dedupe |
| 6 | Null rows, self-transfers, outliers | Flag + tag |

Outputs: `people`, `promotions`, `items`, `transfers`.

In [ ]:
import os, re, json, datetime
import numpy as np
import pandas as pd
from xml.etree import ElementTree as ET

# Works whether Jupyter starts in this folder, in recommendations/, or at the
# repo root. Override with VENMITO_DATA.
DATA = os.environ.get("VENMITO_DATA", "")
if not DATA:
    for candidate in ("raw_data", "../raw_data", "../../recommendations/raw_data",
                      "recommendations/raw_data"):
        if os.path.isdir(candidate):
            DATA = candidate
            break
if not DATA:
    raise FileNotFoundError("Cannot locate raw_data; set VENMITO_DATA.")
print("Reading from:", os.path.abspath(DATA))

## 1. Normalize both people files and outer-join on id

In [ ]:
DEVICE_COLS = ("Android", "Desktop", "Iphone")
CANON = ["id", "first_name", "last_name", "email", "phone", "city", "country", "dob", "devices"]

def _unquote(v):
    return v.strip().strip('"').strip() if isinstance(v, str) else v

def _parse_dob(v):
    """json uses MM/DD/YYYY; yml uses 'May 20, 2000' -- except id 1002."""
    v = _unquote(v)
    for fmt in ("%m/%d/%Y", "%B %d, %Y"):
        try:
            return pd.Timestamp(datetime.datetime.strptime(v, fmt))
        except ValueError:
            pass
    return pd.NaT

def _split_city(v):
    """'Montreal, Canada' -> ('Montreal', 'Canada')."""
    v = _unquote(v)
    city, _, country = v.rpartition(", ")
    return (city.strip(), country.strip()) if city else (v, pd.NA)

def _split_name(v):
    """'Jamie Bright' -> ('Jamie', 'Bright'); multi-word surnames stay intact."""
    first, _, last = _unquote(v).partition(" ")
    return first.strip(), last.strip()

In [ ]:
def load_people_json(path):
    with open(path) as fh:
        raw = json.load(fh)
    return pd.DataFrame([{
        "id": int(p["id"]),                                   # "0001" -> 1
        "first_name": p["first_name"].strip(),
        "last_name": p["last_name"].strip(),
        "email": p["email"].strip().lower(),
        "phone": p["telephone"].strip(),
        "city": p["location"]["City"].strip(),
        "country": p["location"]["Country"].strip(),
        "dob": _parse_dob(p["dob"]),
        "devices": frozenset(d.strip() for d in p["devices"]),
    } for p in raw], columns=CANON)

def load_people_yml(path):
    """Minimal parser for this file's flat 'list of scalar maps' shape.

    Not PyYAML: keeps the dependency surface small and leaves the quoting quirk
    on id 1002 visible instead of silently normalised away.
    """
    recs, cur = [], None
    with open(path) as fh:
        for line in fh:
            line = line.rstrip("\n")
            if line.startswith("- "):
                cur = {}
                recs.append(cur)
                line = " " + line[1:]
            m = re.match(r"\s+([A-Za-z_]+):\s*(.*)$", line)
            if m and cur is not None:
                cur[m.group(1)] = m.group(2)

    rows = []
    for r in recs:
        first, last = _split_name(r["name"])
        city, country = _split_city(r["city"])
        rows.append({
            "id": int(_unquote(r["id"])),
            "first_name": first, "last_name": last,
            "email": _unquote(r["email"]).lower(),
            "phone": _unquote(r["phone"]),
            "city": city, "country": country,
            "dob": _parse_dob(r["dob"]),
            # 3 x 0/1 columns -> the set representation the JSON file uses
            "devices": frozenset(c for c in DEVICE_COLS if _unquote(r[c]) == "1"),
        })
    return pd.DataFrame(rows, columns=CANON)

pj = load_people_json(os.path.join(DATA, "people.json"))
py = load_people_yml(os.path.join(DATA, "people.yml"))
print(f"people.json -> {len(pj)} rows")
print(f"people.yml  -> {len(py)} rows")
pj.head(3)

In [ ]:
merged = pj.merge(py, on="id", how="outer", suffixes=("_json", "_yml"), indicator=True)
VALUE_COLS = [c for c in CANON if c != "id"]
print(merged["_merge"].value_counts().to_string())

# Diff the overlapping rows BEFORE coalescing, so nothing resolves silently.
both = merged[merged["_merge"] == "both"]
mask = pd.Series(False, index=both.index)
for c in VALUE_COLS:
    mask |= both[f"{c}_json"].ne(both[f"{c}_yml"])
conflicts = both[mask]
print(f"\noverlapping ids: {len(both)}   conflicting: {len(conflicts)} -> {sorted(conflicts['id'])}")
for c in VALUE_COLS:
    for _, r in conflicts[conflicts[f"{c}_json"].ne(conflicts[f"{c}_yml"])].iterrows():
        print(f"  id {r['id']}  {c:<11} json={r[f'{c}_json']!r}  yml={r[f'{c}_yml']!r}")

In [ ]:
# Precedence is inert wherever the files agree; it only bites on the ids above.
PRECEDENCE = "people.json"          # or "people.yml"
hi, lo = ("json", "yml") if PRECEDENCE == "people.json" else ("yml", "json")

people = merged[["id"]].copy()
for c in VALUE_COLS:
    people[c] = merged[f"{c}_{hi}"].where(merged[f"{c}_{hi}"].notna(), merged[f"{c}_{lo}"])
people["source"] = merged["_merge"].map(
    {"both": "both", "left_only": "people.json", "right_only": "people.yml"}).astype("string")
people = people.sort_values("id").reset_index(drop=True)
people_conflicts = conflicts[["id"] + [f"{c}_{s}" for c in VALUE_COLS for s in ("json", "yml")]]

print(f"merged population: {len(people)}   id range {people['id'].min()}..{people['id'].max()}")
print(f"gaps: {sorted(set(range(1, people['id'].max()+1)) - set(people['id']))}")
print("\ncountry coverage -- the reason for the outer join:")
print(pd.DataFrame({"json_only": pj["country"].value_counts(),
                    "yml_only": py["country"].value_counts(),
                    "merged": people["country"].value_counts()}
                   ).fillna(0).astype(int).to_string())

## 2. Decide id 998 explicitly

Ground truth from the tail of both files:

| | 998 | 999 | 1002 |
|---|---|---|---|
| `people.json` | Fey Kuser | Fatimah Johns | — |
| `people.yml` | Fatimah Johns | — | Fey Kuser |

yml's 998 is field-identical to json's 999, so **Fatimah is intact at 999** — inserting Fey
shifted her by one in the JSON file only. Two defects: id 998 means different people in
different files, and Fey is duplicated at 998 and 1002 with disjoint natural keys.

In [ ]:
ID_998_POLICY = "json_ids"   # "json_ids" | "quarantine"

people["is_synthetic"] = False
people["canonical_id"] = people["id"]
STALE_IDS, RETIRED_KEYS = {}, {"email": {}, "phone": {}}

yml998 = people_conflicts.loc[people_conflicts["id"] == 998].iloc[0]
json999 = people.loc[people["id"] == 999].iloc[0]
same = all(yml998[f"{c}_yml"] == json999[c] for c in VALUE_COLS)
print(f"yml 998 is field-identical to json 999 (Fatimah Johns): {same}")
assert same, "assumption broken -- re-inspect before applying any policy"

if ID_998_POLICY == "json_ids":
    # Adopt json's id space: 998=Fey, 999=Fatimah. yml's 998 is a shifted
    # reference to 999 and carries nothing json lacks. Collapse Fey 1002 -> 998.
    STALE_IDS = {998: 999}
    people.loc[people["id"].isin((998, 1002)), "is_synthetic"] = True
    people.loc[people["id"] == 1002, "canonical_id"] = 998
    fey_alt = people.loc[people["id"] == 1002].iloc[0]
    RETIRED_KEYS = {"email": {fey_alt["email"]: 998}, "phone": {fey_alt["phone"]: 998}}
elif ID_998_POLICY == "quarantine":
    STALE_IDS = {998: 999}
    quarantined = people[people["id"].isin((998, 1002))].copy()
    people = people[~people["id"].isin((998, 1002))].reset_index(drop=True)
    print(f"quarantined ids {sorted(quarantined['id'])}")
else:
    raise ValueError(ID_998_POLICY)

print(f"\npolicy = {ID_998_POLICY}")
print(f"id rows: {len(people)}   distinct entities: {people['canonical_id'].nunique()}")
print(people[people["id"].isin((998, 999, 1002))]
      [["id", "first_name", "last_name", "email", "phone", "country",
        "is_synthetic", "canonical_id"]].to_string(index=False))
print(f"\nstale id remap (yml -> real): {STALE_IDS}")
print(f"retired keys re-pointed: {RETIRED_KEYS}")

## 3. Identity lookups, then resolve transactions and promotions

In [ ]:
def build_lookup(df, key_col, retired):
    dupes = sorted(df[key_col][df[key_col].duplicated()])
    lut = dict(zip(df[key_col], df["canonical_id"]))
    lut.update(retired.get(key_col, {}))       # aliases win over the live table
    return lut, dupes

phone_to_id, dup_phones = build_lookup(people, "phone", RETIRED_KEYS)
email_to_id, dup_emails = build_lookup(people, "email", RETIRED_KEYS)
print(f"phone -> id : {len(phone_to_id):>4} keys   duplicates: {dup_phones or 'none'}")
print(f"email -> id : {len(email_to_id):>4} keys   duplicates: {dup_emails or 'none'}")
assert not dup_phones and not dup_emails, "ambiguous natural key -- lookup would be lossy"

In [ ]:
# transactions resolve on phone
tx_rows = []
for e in ET.parse(os.path.join(DATA, "transactions.xml")).getroot():
    phone = (e.findtext("phone") or "").strip()
    tx_rows.append({"transaction_id": int(e.get("id")), "phone": phone,
                    "store": e.findtext("store"), "date": pd.to_datetime(e.findtext("date")),
                    "person_id": phone_to_id.get(phone, pd.NA)})
tx = pd.DataFrame(tx_rows).astype({"person_id": "Int64"})
tx["is_orphan"] = tx["person_id"].isna()
print(f"transactions: {len(tx)}   resolved: {(~tx['is_orphan']).sum()}   orphans: {tx['is_orphan'].sum()}")
print(tx.loc[tx["is_orphan"], ["transaction_id", "phone", "store", "date"]].to_string(index=False))

In [ ]:
# promotions resolve on email, falling back to phone
promos = pd.read_csv(os.path.join(DATA, "promotions.csv"), dtype=str, keep_default_na=False)
for c in ("client_email", "telephone"):
    promos[c] = promos[c].str.strip()
promos["client_email"] = promos["client_email"].str.lower()

by_email = promos["client_email"].map(email_to_id).astype("Int64")
by_phone = promos["telephone"].map(phone_to_id).astype("Int64")
promos["person_id"] = by_email.fillna(by_phone)
promos["resolved_via"] = pd.Series("unresolved", index=promos.index, dtype="string")
promos.loc[by_phone.notna(), "resolved_via"] = "phone"
promos.loc[by_email.notna(), "resolved_via"] = "email"
promos["promotion_date"] = pd.to_datetime(promos["promotion_date"])
promos["responded"] = promos["responded"].map({"Yes": True, "No": False})

disagree = promos[by_email.notna() & by_phone.notna() & by_email.ne(by_phone)]
print(f"promotions: {len(promos)}   resolved: {promos['person_id'].notna().sum()}   "
      f"orphans: {promos['person_id'].isna().sum()}")
print(promos["resolved_via"].value_counts().to_string())
print(f"rows where email and phone resolve to DIFFERENT people: {len(disagree)}")

fey = promos[promos["client_email"] == "fey_kuser@example.com"]
print(f"promotions on the retired 'fey_kuser@example.com' key: {len(fey)} "
      f"-> canonical person_id {sorted(set(fey['person_id'].dropna()))}")

## 4. Re-key promotions on row position

In [ ]:
# The CSV `id` is not a key: ids 200-212 each appear twice on unrelated rows.
# Demote it to provenance; key on row position.
promotions = promos.rename(columns={"id": "source_id", "client_email": "email",
                                    "telephone": "phone"}).copy()
promotions.insert(0, "promotion_key", promotions.index.to_numpy())
promotions["source_file"] = "promotions.csv"
promotions["source_row"] = promotions["promotion_key"] + 2      # header + 1-indexed
promotions["source_id_is_ambiguous"] = promotions["source_id"].duplicated(keep=False)
promotions = promotions[["promotion_key", "person_id", "promotion", "responded",
                         "promotion_date", "resolved_via", "email", "phone",
                         "source_id", "source_id_is_ambiguous", "source_file", "source_row"]]

print(f"rows: {len(promotions)}   distinct source_id: {promotions['source_id'].nunique()}"
      f"   distinct promotion_key: {promotions['promotion_key'].nunique()}")
print(f"rows whose source_id is ambiguous: {promotions['source_id_is_ambiguous'].sum()}")
print("\nthe two rows sharing source_id 200 -- now distinguishable:")
print(promotions[promotions["source_id"] == "200"].to_string(index=False))

## 5. Transactions: recompute price, flag zero/negative, dedupe

Bad rows are **flagged, not dropped**. `price_reported` is kept beside the recomputed
`price` so the correction stays auditable.

In [ ]:
# One row per line item. Note the <item><item>NAME</item></item> nesting --
# iterate children of <items> rather than using .//item, which double-counts.
li = []
for e in ET.parse(os.path.join(DATA, "transactions.xml")).getroot():
    tid, phone = int(e.get("id")), (e.findtext("phone") or "").strip()
    for pos, it in enumerate(e.find("items")):
        li.append({"transaction_id": tid, "line_no": pos, "phone": phone,
                   "store": e.findtext("store"), "date": pd.to_datetime(e.findtext("date")),
                   "item": it.findtext("item"),
                   "price_reported": float(it.findtext("price")),
                   "price_per_item": float(it.findtext("price_per_item")),
                   "quantity": float(it.findtext("quantity"))})
items = pd.DataFrame(li)
items["person_id"] = items["phone"].map(phone_to_id).astype("Int64")
items["is_orphan"] = items["person_id"].isna()

items["price"] = items["price_per_item"] * items["quantity"]
items["price_mismatch"] = ~np.isclose(items["price_reported"], items["price"], atol=0.011)
items["price_zero"] = items["price_reported"].eq(0)
items["price_negative"] = items["price_reported"].lt(0)
items["needs_review"] = items[["price_mismatch", "price_zero", "price_negative"]].any(axis=1)

print(f"line items: {len(items)} across {items['transaction_id'].nunique()} transactions")
print(f"flagged (NOT dropped): mismatch={items['price_mismatch'].sum()} "
      f"zero={items['price_zero'].sum()} negative={items['price_negative'].sum()}")
print(items.loc[items["needs_review"],
                ["transaction_id", "item", "price_reported", "price_per_item", "quantity",
                 "price", "price_mismatch", "price_zero", "price_negative"]].to_string(index=False))

In [ ]:
# Dedupe by content signature, not by hardcoding ids -- survives regeneration.
basket = (items.sort_values(["transaction_id", "item"]).groupby("transaction_id")
          .apply(lambda g: tuple(zip(g["item"], g["quantity"], g["price"])), include_groups=False))
sig = pd.DataFrame({"basket": basket}).join(
    items.groupby("transaction_id")[["phone", "store", "date"]].first())
sig["signature"] = list(zip(sig["phone"], sig["store"], sig["date"], sig["basket"]))
sig["is_duplicate"] = sig["signature"].duplicated(keep="first")     # keep earliest id
sig["duplicate_of"] = sig["signature"].map(
    sig[~sig["is_duplicate"]].reset_index().set_index("signature")["transaction_id"])
sig.loc[~sig["is_duplicate"], "duplicate_of"] = pd.NA

items = items.merge(sig[["is_duplicate", "duplicate_of"]], on="transaction_id", how="left")
items["is_clean"] = ~items[["needs_review", "is_duplicate", "is_orphan"]].any(axis=1)

print("duplicate transactions flagged:")
print(sig[sig["is_duplicate"]].reset_index()[["transaction_id", "duplicate_of", "store", "date"]]
      .to_string(index=False))
print(f"\nline items: {len(items)}   clean: {items['is_clean'].sum()}   "
      f"flagged: {(~items['is_clean']).sum()}   dropped: 0")
print(f"gross revenue  all rows        : {items['price'].sum():>10,.2f}")
print(f"gross revenue excl. duplicate  : {items.loc[~items['is_duplicate'], 'price'].sum():>10,.2f}")

## 6. Transfers: flag nulls, self-transfers, and tag outliers

Read raw so the null rows survive. The outage dates are preserved as a data-quality note
rather than discarded, and the outliers are tagged — they are the fraud signal, not noise.

In [ ]:
transfers = pd.read_csv(os.path.join(DATA, "transfers.csv"), dtype=str, keep_default_na=False)
transfers.insert(0, "transfer_key", transfers.index.to_numpy())
for c in ("sender_id", "recipient_id", "amount", "date"):
    transfers[c] = transfers[c].str.strip()

# (a) null rows -> ingestion outage
transfers["is_null_row"] = transfers["sender_id"].eq("") & transfers["recipient_id"].eq("")
transfers["date"] = pd.to_datetime(transfers["date"])
transfers["amount"] = pd.to_numeric(transfers["amount"])

outage_dates = (transfers.loc[transfers["is_null_row"]]
                .groupby("date").size().rename("null_rows").reset_index())
print(f"null rows flagged (not dropped): {transfers['is_null_row'].sum()}")
print("\noutage calendar -- keep as a data-quality note:")
print(outage_dates.to_string(index=False))

In [ ]:
# ids canonicalised. STALE_IDS is a people.yml-local remap and is NOT applied here:
# transfers.csv has no file-of-origin, so its 998 is ambiguous -- flag, don't resolve.
canon = dict(zip(people["id"], people["canonical_id"]))
for c in ("sender_id", "recipient_id"):
    transfers[c] = pd.to_numeric(transfers[c].replace("", pd.NA)).astype("Int64")
    transfers[c] = transfers[c].map(lambda i: canon.get(i, i) if pd.notna(i) else i).astype("Int64")
transfers["is_ambiguous_998"] = transfers[["sender_id", "recipient_id"]].eq(998).any(axis=1)

# (b) self-transfers
transfers["is_self_transfer"] = (transfers["sender_id"].eq(transfers["recipient_id"])
                                 & transfers["sender_id"].notna())
print(f"self-transfers flagged: {transfers['is_self_transfer'].sum()}")
print(transfers.loc[transfers["is_self_transfer"]]
      .groupby(["sender_id", "recipient_id"]).size().rename("n").to_string())

In [ ]:
# (c) outliers TAGGED, never deleted
real = transfers.loc[~transfers["is_null_row"]]
q1, q3 = real["amount"].quantile([0.25, 0.75])
fence = q3 + 3 * (q3 - q1)                  # Tukey far-out fence, robust to the plants
print(f"IQR far-out fence: {fence:,.2f}  (q1={q1:.2f} q3={q3:.2f} median={real['amount'].median():.2f})")

transfers["is_amt_outlier"] = transfers["amount"].gt(fence) & ~transfers["is_null_row"]
transfers["is_round_amount"] = (transfers["amount"].mod(1).eq(0) & transfers["amount"].ge(100)
                                & ~transfers["is_null_row"])

# reciprocal round-trips: A->B and B->A, same amount, within 7 days
pairs = real.assign(a=real[["sender_id", "recipient_id"]].min(axis=1),
                    b=real[["sender_id", "recipient_id"]].max(axis=1))
recip = set()
for _, g in pairs.groupby(["a", "b", "amount"]):
    if g["sender_id"].nunique() > 1 and g["date"].max() - g["date"].min() <= pd.Timedelta("7D"):
        recip |= set(g["transfer_key"])
transfers["is_reciprocal_pair"] = transfers["transfer_key"].isin(recip)

# same-day fan-out: one sender, >=3 distinct recipients on a single date
fan = real.groupby(["sender_id", "date"])["recipient_id"].nunique().loc[lambda s: s >= 3].index
transfers["is_fanout"] = pd.MultiIndex.from_frame(transfers[["sender_id", "date"]]).isin(fan)

In [ ]:
FLAGS = ["is_null_row", "is_self_transfer", "is_amt_outlier", "is_round_amount",
         "is_reciprocal_pair", "is_fanout", "is_ambiguous_998"]
transfers["flags"] = ["|".join(f[3:] for f in FLAGS if r[f]) for _, r in transfers[FLAGS].iterrows()]
transfers["is_clean"] = transfers["flags"].eq("")

print(transfers[FLAGS].sum().rename("n").to_string())
print(f"\nrows: {len(transfers)}   clean: {transfers['is_clean'].sum()}   "
      f"flagged: {(~transfers['is_clean']).sum()}   dropped: 0")
print("\nflagged non-null rows above the fence or matching a pattern:")
print(transfers.loc[~transfers["is_clean"] & ~transfers["is_null_row"] & ~transfers["is_self_transfer"],
                    ["transfer_key", "sender_id", "recipient_id", "amount", "date", "flags"]]
      .to_string(index=False))

## Result

In [ ]:
analysis = transfers[transfers["is_clean"]]
print(f"people      {len(people):>5} rows   {people['canonical_id'].nunique()} distinct entities")
print(f"promotions  {len(promotions):>5} rows   {promotions['person_id'].notna().sum()} resolved")
print(f"items       {len(items):>5} rows   {items['is_clean'].sum()} clean")
print(f"transfers   {len(transfers):>5} rows   {analysis.shape[0]} clean")
print(f"\ntransfers analysis view: median {analysis['amount'].median():.2f}, "
      f"max {analysis['amount'].max():.2f}")

# --- post-conditions ------------------------------------------------------
assert len(people) == 1002 and people["canonical_id"].nunique() == 1001
assert people["country"].value_counts()["France"] == 42
assert people["country"].value_counts()["Spain"] == 26
assert people.loc[people.id == 999, "last_name"].iloc[0] == "Johns"
assert tx["is_orphan"].sum() == 4
assert promotions["person_id"].notna().all() and promotions["promotion_key"].is_unique
assert items["price_mismatch"].sum() == 2 and items["is_duplicate"].sum() == 2
assert len(transfers) == 614 and transfers["is_null_row"].sum() == 15
assert transfers["is_self_transfer"].sum() == 56
print("\nall post-conditions passed")

In [ ]:
OUT = os.path.join(os.path.dirname(DATA) or ".", "data_clean")
os.makedirs(OUT, exist_ok=True)
people.assign(devices=people["devices"].map(lambda s: "|".join(sorted(s)))
              ).to_csv(f"{OUT}/people.csv", index=False)
promotions.to_csv(f"{OUT}/promotions.csv", index=False)
items.to_csv(f"{OUT}/transaction_items.csv", index=False)
transfers.to_csv(f"{OUT}/transfers.csv", index=False)
outage_dates.to_csv(f"{OUT}/transfers_outage_dates.csv", index=False)
print("wrote:", sorted(os.listdir(OUT)))